# KnoverseAI Cognitive Mechanics Benchmark

**Competition:** Measuring Progress Toward AGI: Cognitive Abilities

This notebook uses paired human-AI telemetry from two games to evaluate whether an LLM can infer cognitive architecture from behavioral data. The diagnostic framework measures decision-making PROCESS -- not outcomes.

## Architecture
- **Layer A:** Visualization of the AI iteration arc (v1.0 -> v2.4) showing how telemetry-driven development produced increasingly human-like behavioral traces
- **Layer B:** kbench benchmark task -- feed raw telemetry to a model, ask it to diagnose the player's cognitive profile, score against ground truth metrics

## Dataset
- 10 Pathogenika sessions (Unity RTS: human vs AI playing immune system cells)
- 4 PetroActive sessions (Three.js mech exploration with radar waypoint navigation)
- Session annotations with corrected team labels and AI version history

In [ ]:
import json
import math
import os
import glob
from collections import Counter
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

# Kaggle dataset path (flat structure, prefixed filenames)
DATA_DIR = Path('/kaggle/input/datasets/robertclegg/knoverseai-cognitive-mechanics-telemetry')
if not DATA_DIR.exists():
    DATA_DIR = Path('/kaggle/input/knoverseai-cognitive-mechanics-telemetry')
if not DATA_DIR.exists():
    DATA_DIR = Path('dataset-flat')  # local fallback

path_sessions = sorted(glob.glob(str(DATA_DIR / 'pathogenika_behavioral_*.json')))
petro_sessions = sorted(glob.glob(str(DATA_DIR / 'petroactive_petro_*.json')))

print(f'Data directory: {DATA_DIR}')
print(f'Pathogenika sessions: {len(path_sessions)}')
print(f'PetroActive sessions: {len(petro_sessions)}')

## Load Data

In [ ]:
# Load benchmark CSV
csv_path = DATA_DIR / 'benchmark_results.csv'
df = pd.read_csv(csv_path)
print(f'Loaded {len(df)} sessions from benchmark_results.csv')
df[['game', 'session', 'aiVer', 'humanTeamName', 'aiTeamName', 'duration', 'total_events']].head(14)

In [ ]:
# Load annotations
with open(DATA_DIR / 'pathogenika_session_annotations.json') as f:
    path_annotations = json.load(f)
with open(DATA_DIR / 'petroactive_session_annotations.json') as f:
    petro_annotations = json.load(f)

print(f'Pathogenika annotations: {len(path_annotations["sessions"])} sessions')
print(f'PetroActive annotations: {len(petro_annotations["sessions"])} sessions')

---
# Layer A: The AI Iteration Arc

The AI opponent evolved from v1.0 to v2.4 across 7 sessions, driven entirely by telemetry analysis. Each version fixed bugs identified from the behavioral data.

In [ ]:
# Filter to Pathogenika AI sessions only
ai_sessions = df[(df['game'] == 'Pathogenika') & (df['aiVer'].notna()) & (df['aiVer'] != 'pre-AI')].copy()
ai_sessions = ai_sessions.sort_values('session').reset_index(drop=True)
ai_sessions['session_num'] = range(1, len(ai_sessions) + 1)
ai_sessions['ai_version_label'] = ai_sessions['aiVer'].fillna('none')
print(f'{len(ai_sessions)} AI sessions loaded')
print(ai_sessions[['session_num', 'ai_version_label', 'humanTeamName', 'duration', 'total_events', 'ai_active_pct']].to_string(index=False))

In [ ]:
# Plot 1: AI Behavioral Evolution
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('AI Behavioral Evolution: v1.0 -> v2.4', fontsize=16, fontweight='bold', color='#00e5ff')
fig.patch.set_facecolor('#0a0f14')

colors_human = '#00e5ff'
colors_ai = '#FFD700'

for ax in axes.flat:
    ax.set_facecolor('#0a0f14')
    ax.tick_params(colors='white')
    ax.xaxis.label.set_color('white')
    ax.yaxis.label.set_color('white')
    ax.title.set_color('white')
    for spine in ax.spines.values():
        spine.set_color('#333')

ax = axes[0, 0]
ax.bar(ai_sessions['session_num'], ai_sessions['ai_active_pct'], color=colors_ai, alpha=0.8)
ax.axhline(y=100, color=colors_human, linestyle='--', alpha=0.5, label='Human (100%)')
ax.set_title('AI Active Duration %')
ax.set_xlabel('Session')
ax.set_ylabel('%')
ax.set_ylim(0, 110)
ax.legend(facecolor='#0a0f14', edgecolor='#333', labelcolor='white')

ax = axes[0, 1]
ax.bar(ai_sessions['session_num'], ai_sessions['ai_switch_rate'], color=colors_ai, alpha=0.8, label='AI')
ax.bar(ai_sessions['session_num'], ai_sessions['human_switch_rate'], color=colors_human, alpha=0.4, label='Human')
ax.set_title('Switch Rate (/min)')
ax.set_xlabel('Session')
ax.legend(facecolor='#0a0f14', edgecolor='#333', labelcolor='white')

ax = axes[0, 2]
ax.plot(ai_sessions['session_num'], ai_sessions['ai_entropy'], 'o-', color=colors_ai, label='AI', linewidth=2)
ax.plot(ai_sessions['session_num'], ai_sessions['human_entropy'], 's--', color=colors_human, label='Human', linewidth=2)
ax.set_title('Action Diversity (Shannon Entropy)')
ax.set_xlabel('Session')
ax.set_ylabel('bits')
ax.legend(facecolor='#0a0f14', edgecolor='#333', labelcolor='white')

ax = axes[1, 0]
ax.bar(ai_sessions['session_num'], ai_sessions['ai_ability_rate'], color=colors_ai, alpha=0.8, label='AI')
ax.bar(ai_sessions['session_num'], ai_sessions['human_ability_rate'], color=colors_human, alpha=0.4, label='Human')
ax.set_title('Ability Rate (/min)')
ax.set_xlabel('Session')
ax.legend(facecolor='#0a0f14', edgecolor='#333', labelcolor='white')

ax = axes[1, 1]
ax.bar(ai_sessions['session_num'], ai_sessions['ai_wp_success_pct'], color=colors_ai, alpha=0.8, label='AI')
ax.bar(ai_sessions['session_num'], ai_sessions['human_wp_success_pct'], color=colors_human, alpha=0.4, label='Human')
ax.set_title('Waypoint Success %')
ax.set_xlabel('Session')
ax.set_ylabel('%')
ax.legend(facecolor='#0a0f14', edgecolor='#333', labelcolor='white')

ax = axes[1, 2]
ax.plot(ai_sessions['session_num'], ai_sessions['ai_events_per_sec'], 'o-', color=colors_ai, label='AI', linewidth=2)
ax.plot(ai_sessions['session_num'], ai_sessions['human_events_per_sec'], 's--', color=colors_human, label='Human', linewidth=2)
ax.set_title('Decision Tempo (events/sec)')
ax.set_xlabel('Session')
ax.legend(facecolor='#0a0f14', edgecolor='#333', labelcolor='white')

plt.tight_layout()
plt.savefig('ai_evolution.png', dpi=150, facecolor='#0a0f14')
plt.show()

### Key Findings

| Metric | AI v1.0 | AI v2.4 | Human Avg | Trend |
|--------|---------|---------|-----------|-------|
| Active Duration | 12-43% | 98% | 100% | Fixed |
| Cell Switches | 0-1 | 23 | 13 | Matched |
| Unit Types Used | 1-2 | 5 | 3-5 | Matched |
| Ability Uses | 0 | 25 | 4-25 | Fixed |
| Waypoint Outcomes | 0 | 23 | 4-60 | Fixed |
| Node Score | 2-3 | 5 | 6-10 | Improving |
| Action Diversity | 1.0 bits | 1.5 bits | 2.0 bits | Gap remains |

In [ ]:
# Plot 2: Human vs AI event type distribution for the cleanest session
clean_session_file = DATA_DIR / 'pathogenika_behavioral_telemetry_2026-03-27_05-00-13pm.json'
with open(clean_session_file) as f:
    session = json.load(f)

human_types = Counter()
ai_types = Counter()
for e in session['events']:
    if e.get('playerType') == 'ai':
        ai_types[e['eventType']] += 1
    else:
        human_types[e['eventType']] += 1

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Event Distribution -- v2.1.1 Clean Session', fontsize=13, color='#00e5ff')
fig.patch.set_facecolor('#0a0f14')

for ax in [ax1, ax2]:
    ax.set_facecolor('#0a0f14')

h_labels = list(human_types.keys())
h_sizes = list(human_types.values())
cyan_shades = plt.cm.cool(np.linspace(0.3, 0.9, len(h_labels)))
ax1.pie(h_sizes, labels=h_labels, autopct='%1.0f%%', colors=cyan_shades, textprops={'fontsize': 8, 'color': 'white'})
ax1.set_title('Human Events', color='#00e5ff')

a_labels = list(ai_types.keys())
a_sizes = list(ai_types.values())
gold_shades = plt.cm.autumn(np.linspace(0.2, 0.8, len(a_labels)))
ax2.pie(a_sizes, labels=a_labels, autopct='%1.0f%%', colors=gold_shades, textprops={'fontsize': 8, 'color': 'white'})
ax2.set_title('AI Events', color='#FFD700')

plt.tight_layout()
plt.show()

---
# Layer B: Cognitive Mechanics Benchmark Task

Can an LLM infer a player's cognitive architecture from their behavioral telemetry alone?

The task: given raw telemetry JSON, diagnose the player's:
1. **Decision tempo** -- how fast do they act?
2. **Action diversity** -- how varied is their behavior?
3. **Strategic adaptation** -- do they change approach over time?
4. **Role understanding** -- do they use units according to their function?
5. **Player type** -- is this a human or an AI?

In [ ]:
import kaggle_benchmarks as kbench
from kaggle_benchmarks import assertions

In [ ]:
def load_session_for_task(filename):
    """Load a Pathogenika session and prepare it for the benchmark task."""
    filepath = DATA_DIR / filename
    with open(filepath) as f:
        session = json.load(f)
    
    events = session.get('events', [])
    duration = session.get('matchDuration', 1)
    
    human_events = [e for e in events if e.get('playerType') != 'ai']
    ai_events = [e for e in events if e.get('playerType') == 'ai']
    
    def compute_entropy(evts):
        types = Counter(e['eventType'] for e in evts)
        total = sum(types.values())
        if total == 0: return 0.0
        return -sum((c/total) * math.log2(c/total) for c in types.values() if c > 0)
    
    ground_truth = {
        'has_ai': len(ai_events) > 0,
        'human_events_per_sec': round(len(human_events) / duration, 2),
        'human_entropy': round(compute_entropy(human_events), 2),
        'human_switches': sum(1 for e in human_events if e['eventType'] == 'SwitchEvent'),
        'ai_events_per_sec': round(len(ai_events) / duration, 2) if ai_events else 0,
        'ai_entropy': round(compute_entropy(ai_events), 2) if ai_events else 0,
        'ai_switches': sum(1 for e in ai_events if e['eventType'] == 'SwitchEvent'),
        'duration': round(duration),
        'total_events': len(events),
    }
    
    sample_events = events[:50]
    session_summary = {
        'sessionId': session.get('sessionId'),
        'matchDuration': duration,
        'totalEvents': len(events),
        'playerTeam': session.get('playerTeam'),
        'sample_events': sample_events
    }
    
    return session_summary, ground_truth

In [ ]:
# Prepare evaluation data -- use clean sessions only (flat filenames with pathogenika_ prefix)
eval_sessions = [
    'pathogenika_behavioral_telemetry_2026-03-27_05-00-13pm.json',
    'pathogenika_behavioral_telemetry_2026-03-27_05-19-42pm.json',
    'pathogenika_behavioral_telemetry_2026-03-27_05-36-20pm.json',
]

eval_data = []
for fname in eval_sessions:
    summary, truth = load_session_for_task(fname)
    eval_data.append({
        'session_file': fname,
        'session_json': json.dumps(summary, indent=2)[:8000],
        'ground_truth': json.dumps(truth),
        **truth
    })

eval_df = pd.DataFrame(eval_data)
print(f'Prepared {len(eval_df)} sessions for evaluation')
eval_df[['session_file', 'has_ai', 'human_entropy', 'ai_entropy', 'duration']]

In [ ]:
@kbench.task(name='cognitive_diagnosis')
def cognitive_diagnosis(llm, session_json, ground_truth, **kwargs):
    """Feed raw telemetry to the model and ask it to diagnose cognitive architecture."""
    
    prompt = f"""You are a cognitive scientist analyzing behavioral telemetry from a real-time strategy game.

The game is Pathogenika -- players control immune system cells fighting pathogen invaders.
Two player types exist: "human" (keyboard/mouse) and "ai" (rule-based agent).

Below is a telemetry session with the first 50 events. Each event has:
- eventType: what happened
- timestamp: seconds since match start
- playerType: "human" or "ai" (may be missing in older sessions)
- data: event-specific fields

SESSION DATA:
{session_json}

ANALYZE THIS SESSION AND ANSWER:

1. DECISION TEMPO: What is the approximate events-per-second rate for each player type? Is the human faster or slower than the AI?

2. ACTION DIVERSITY: Which player uses a wider variety of event types? Estimate the Shannon entropy (in bits) for each player's event distribution.

3. STRATEGIC ADAPTATION: Look at the sequence of events. Does either player change their behavior pattern over time? Describe any shifts.

4. ROLE UNDERSTANDING: Different cell types have different functions (VirusType captures nodes, NKCellType fires projectiles at nodes, NeutrophilType fights in melee). Does the human player show evidence of understanding these roles based on their unit switching patterns?

5. PLAYER IDENTIFICATION: Based on the behavioral patterns alone, which stream shows more human-like cognition and which shows more machine-like behavior? What specific evidence supports your classification?

Provide specific numbers where possible. Reference event types and timestamps from the data."""
    
    response = llm.prompt(prompt)
    
    truth = json.loads(ground_truth)
    
    criteria = [
        f"The response should identify that the human decision rate is approximately {truth['human_events_per_sec']} events/sec",
        f"The response should identify that the AI decision rate is approximately {truth['ai_events_per_sec']} events/sec",
        f"The response should note that human action diversity (entropy ~{truth['human_entropy']} bits) is higher than AI ({truth['ai_entropy']} bits)",
        "The response should correctly identify which stream is human and which is AI based on behavioral patterns",
        "The response should mention specific cell types and their roles (e.g., VirusType for node capture, NeutrophilType for combat)",
        "The response should identify evidence of strategic adaptation or learning in the human player's behavior"
    ]
    
    assessment = assertions.assess_response_with_judge(
        criteria=criteria,
        response_text=response,
        judge_llm=kbench.judge_llm
    )
    
    return assessment

In [ ]:
# Run evaluation across all clean sessions
results = cognitive_diagnosis.evaluate(
    llm=[kbench.llm],
    evaluation_data=eval_df
)
results

---
## Methodology Notes

**Why behavioral telemetry as a cognitive benchmark?**

Traditional AGI benchmarks test knowledge retrieval or reasoning in isolation. This benchmark tests whether a model can:
1. Parse real-world behavioral data (not synthetic)
2. Infer latent cognitive states from observable actions
3. Distinguish human from machine decision-making
4. Identify learning and adaptation patterns

The telemetry comes from actual gameplay sessions where a human and a rule-based AI played the same game simultaneously. The AI's behavioral trace was designed to be structurally identical to the human's (same event types, same action pipeline) -- the only difference is the cognitive architecture behind the decisions.

**Ground truth** is derived from computed metrics (Shannon entropy, event rates, switch intervals) rather than human labels, making the benchmark reproducible and objective.

**Prior research** on this approach: Kebritchi, Hirumi & Bai (UCF, 2008) demonstrated that game-based learning produces measurable knowledge transfer. The KnoverseAI framework extends this by capturing the cognitive process itself, not just the outcome.

In [ ]:
%choose cognitive_diagnosis